In [0]:
# 1. Carga del archivo desde el Volume
path_json = "/Volumes/test/data_api_pib/data_api_volumen_1/gdp_raw_20260318_1441.json"

# Forzamos la lectura multilínea para que respete el formato de Array [ ... ]
df = (spark.read
      .option("multiLine", "true")
      .option("mode", "PERMISSIVE") # Evita que explote si hay un error pequeño
      .json(path_json))

# Si el JSON tiene la estructura típica del Banco Mundial, 
# la primera fila suele ser metadata y la segunda los datos reales.
if "_corrupt_record" in df.columns:
    print("❌ Seguimos teniendo registros corruptos. Probemos leyéndolo como texto para inspeccionar:")
    display(spark.read.text(path_json).limit(5))
else:
    print(f"✅ ¡Logrado! Columnas detectadas: {df.columns}")
    display(df)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, MapType

# 1. Definimos el esquema manualmente para que Spark no tenga que "adivinar"
# Usamos MapType para las columnas anidadas (como country e indicator) que son diccionarios
schema = StructType([
    StructField("country", MapType(StringType(), StringType()), True),
    StructField("countryiso3code", StringType(), True),
    StructField("date", StringType(), True),
    StructField("indicator", MapType(StringType(), StringType()), True),
    StructField("unit", StringType(), True),
    StructField("obs_status", StringType(), True),
    StructField("value", DoubleType(), True) # <-- Forzamos que sea Double siempre
])

try:
    # 2. Creamos el DataFrame usando el esquema definido
    df_gdp = spark.createDataFrame(datos_reales, schema=schema)
    
    print("✅ ¡Esquema aplicado con éxito! Ya no hay conflicto de tipos.")
    
    # 3. Aplanamos de una vez para que veas los datos limpios
    df_final = df_gdp.select(
        col("country.value").alias("pais"),
        col("date").alias("anio"),
        col("value").alias("pib_valor")
    ).filter(col("pib_valor").isNotNull())
    
    display(df_final)

except Exception as e:
    print(f"❌ Error: {e}")